python code to read the weather data


In [18]:
import urllib.request
import json
import sys
import logging
# from config import Config


In [ ]:
# fetching weather data from weather api of https://www.visualcrossing.com/
# define the API key and base URL

In [19]:
api_key ="KMRF8C6UADP84FUDRSGDYBZ2C"
base_url =  "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/"
print (api_key)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)  # set minimum level

# Create a file handler
file_handler = logging.FileHandler('app_log.log')  # log file path
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.info("This is a log message WEATHER_DATA ETL")


INFO:__main__:This is a log message WEATHER_DATA ETL


KMRF8C6UADP84FUDRSGDYBZ2C


In [15]:
# define the location and parameters for the API call
location = "London,UK"
unit_group = "metric"
content_type = "json"
dsn_hostname = "54a2f15b-5c0f-46df-8954-7e38e612c2bd.c1ogj3sd0tgtu0lqde00.databases.appdomain.cloud"
dsn_uid = "MGV47022"
dsn_pwd = "VwteXwlQNaJgq8xe"
dsn_driver = "{IBM DB2 ODBC DRIVER}"
dsn_database = "bludb"
dsn_port = "32733"
dsn_protocol = "TCPIP"
dsn_security = "SSL"
dsn = (
    "DRIVER={0};"
    "DATABASE={1};"
    "HOSTNAME={2};"
    "PORT={3};"
    "PROTOCOL={4};"
    "UID={5};"
    "PWD={6};"
    "SECURITY={7};").format(dsn_driver, dsn_database, dsn_hostname, dsn_port, dsn_protocol, dsn_uid, dsn_pwd,dsn_security)

In [ ]:
#construct the request URL
""" if external config file is used then
from config import Config
 &key={Config.VISUALCROSSING_API_KEY}
"""
# request_url = f"{base_url}{location}?unit_group={unit_group}&include=days&contentType={content_type}&key={api_key}"
# print(request_url)
# with urllib.request.urlopen(request_url) as response:
#   data = response.read()
#   weather_data = json.loads(data.decode('utf-8'))
# # print(json.dumps(weather_data, indent=4))

https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/London,UK?unit_group=metric&include=days&contentType=json&key=KMRF8C6UADP84FUDRSGDYBZ2C


In [5]:
# EXTRACT: fetch weather data for given location from date ragne from visual Crossing API date in YYYY-MM-DD format
def retrieve_weather(city, start_date, end_date):
    request_url = f"{base_url}{city}/{start_date}/{end_date}?unit_group={unit_group}&include=days&contentType={content_type}&key={api_key}"
    print(request_url)
    try:
  # Make the request to the visual Crossing Weather API
      with urllib.request.urlopen(request_url) as response:
        if response.status == 200:
          # Read and decode the response data
          data = response.read()
          # parse JSON data
          weather_data = json.loads(data.decode('utf-8'))

          # example: print weather data
          # print("Weather data for : ",city)
          # print(json.dumps(weather_data, indent =4))
          # dump to external file weather_data.json to be consumed by tranformation.py
          with open("weather_data.json","w") as file:
            json.dump(weather_data, file, indent =4)
          print(" file saved to weather_data.json")
          logger.info(" file saved to weather_data.json")
        else:
          print(f"Error: Received status code {response.state}", file = sys.stderr)
    except urllib.error.URLError as e:
      print(f"Failed to retrieve weather data: {e.reason}", file = sys.stderr)
      logger.error(f"Failed to retrieve weather data: {e.reason}", file = sys.stderr)
    return weather_data

# retrieve_weather("London,UK","2026-03-16","2026-03-16")
# retrieve_weather("FortWorth,USA","2026-03-17","2026-03-17")

In [6]:
# TRANSFORM: code for cleaning and transforming the data
import pandas as pd
import json
import sys
# from config import config

def transform_data(file_name = "weather_data.json"):
# load json file from extraction
  try:
    with open(file_name,"r") as file:
      data = json.load(file)
  except FileNotFoundError:
    print(f"File {file_name} not found.")
    sys.exit(1)
  except json.JSONDecodeError:
    print("Error decoding JSON from file")
    sys.exit(1)
# transform data
  try:
    resolvedAddress = data.get("resolvedAddress", "")
    # print(resolvedAddress)
    records = data.get("days", [])
    # print(records)
    df = pd.DataFrame(
        records,
        columns=["datetime", "temp", "feelslike", "humidity", "precip", "windspeed"]
        )
    df["resolvedAddress"] = resolvedAddress
    # print(df.head())
    # unit conversion
    df.columns = ["date", "temperature", "feels_like", "humidity", "precipitation", "windspeed", "resolvedAddress"]
    df["date"] = pd.to_datetime(df["date"])
    df["temperature"] = pd.to_numeric(df["temperature"], errors='coerce') # if value cannot be converted 'coerce' replaces with "NaN"
    df["feels_like"] = pd.to_numeric(df["feels_like"], errors='coerce')
    df["humidity"] = pd.to_numeric(df["humidity"], errors='coerce')
    df["precipitation"] = pd.to_numeric(df["precipitation"], errors='coerce')
    df["windspeed"] = pd.to_numeric(df["windspeed"], errors='coerce')

    # clean column names
    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
    return df
  except KeyError as e:
    print(f"Missing expected key in data: {e}")
    logger.error(f"Missing expected key in data: {e}")
    sys.exit(1)
  except Exception as e:
    print(f"An error occurred during data transformation: {e}")
    logger.error(f"An error occurred during data transformation: {e}")
    sys.exit(1)

retrieve_weather("FortWorth,USA","2026-03-17","2026-03-17")
transformed_data = transform_data("weather_data.json")
transformed_data.to_csv("transformed_weather_data.csv", index=False)
print(transformed_data)

https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/FortWorth,USA/2026-03-17/2026-03-17?unit_group=metric&include=days&contentType=json&key=KMRF8C6UADP84FUDRSGDYBZ2C


INFO:__main__: file saved to weather_data.json


 file saved to weather_data.json
        date  temperature  feels_like  humidity  precipitation  windspeed  \
0 2026-03-17         48.5        47.0      33.7            0.0       19.2   

                 resolvedaddress  
0  Fort Worth, TX, United States  


In [6]:
!pip install --force-reinstall ibm_db ibm_db_sa
!pip install --upgrade ibm_db ibm_db_sa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.9/613.9 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: greenlet
    Found existing installation: greenlet 3.3.2
    Uninstalling greenlet-3.3.2:
      Successfully uninstalled greenlet-3.3.2
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.48
    Uninstalling SQLAlchemy-2.0.48:
      Successfully uninstalled SQLAlchemy-2.0.48


In [9]:
import pandas as pd
import ibm_db
import ibm_db_dbi
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

# Load transformed data .csv and write to database
def load_data_to_db2(df, table_name="WEATHER_DATA"):
  # connect to IBM DB2 cloud db
  #Replace the placeholder values with your actual Db2 hostname, username, and password:
  dsn_hostname = "54a2f15b-5c0f-46df-8954-7e38e612c2bd.c1ogj3sd0tgtu0lqde00.databases.appdomain.cloud" # e.g.: "54a2f15b-5c0f-46df-8954-7e38e612c2bd.c1ogj3sd0tgtu0lqde00.databases.appdomain.cloud"
  dsn_uid = "MGV47022"        # e.g. "abc12345"
  dsn_pwd = "VwteXwlQNaJgq8xe"      # e.g. "7dBZ3wWt9XN6$o0J"

  dsn_driver = "{IBM DB2 ODBC DRIVER}"
  dsn_database = "bludb"            # e.g. "BLUDB"
  dsn_port = "32733"                # e.g. "32733"
  dsn_protocol = "TCPIP"            # i.e. "TCPIP"
  dsn_security = "SSL"              #i.e. "SSL"
  dsn = (
      "DRIVER={0};"
      "DATABASE={1};"
      "HOSTNAME={2};"
      "PORT={3};"
      "PROTOCOL={4};"
      "UID={5};"
      "PWD={6};"
      "SECURITY={7};").format(dsn_driver, dsn_database, dsn_hostname, dsn_port, dsn_protocol, dsn_uid, dsn_pwd,dsn_security)

  class PatchedConnection(ibm_db_dbi.Connection):
    @property
    def current_schema(self):
        return dsn_uid

  def get_conn():
      conn = ibm_db.connect(dsn, "", "")
      return PatchedConnection(conn)

  try:
    conn = ibm_db.connect(dsn, "", "")
    # engine = create_engine(dsn)
    # engine = create_engine(f"db2+ibm_db://{dsn_uid}:{dsn_pwd}@{dsn_hostname}:{dsn_port}/{dsn_database}?security=SSL")
    engine = create_engine("db2+ibm_db://", creator=get_conn)
    print ("Connected to database: ", dsn_database, "as user: ", dsn_uid, "on host: ", dsn_hostname)
    server = ibm_db.server_info(conn)

    print ("DBMS_NAME: ", server.DBMS_NAME)
    print ("DBMS_VER:  ", server.DBMS_VER)
    print ("DB_NAME:   ", server.DB_NAME)
    # create table in remote IBM cloud DB2 database
    # createQuery = "create table weather_data(DATE DATE, temperature NUMERIC(10, 2), feels_like NUMERIC(10, 2), humidity NUMERIC(10, 2), windspeed NUMERIC(10, 2), resolvedaddress VARCHAR(50))"
    #Now fill in the name of the method and execute the statement
    # createStmt = ibm_db.exec_immediate(conn,createQuery)
    df.to_sql(table_name, engine, if_exists="replace", index=False,chunksize=1000)
    print(f"Data loaded into the table {table_name} successfully!")
    logger.info(f"Data loaded into the table {table_name} successfully!")

    ibm_db.close(conn)
  except SQLAlchemyError as e:
    print(f"SQLAlchemyError : {str(e)} ", )
  except Exception as e:
    print(f"Error loading data to IBMDB2 : {e}" )
    print(f"Error loading data to ibm_db.conn_errormsg(): {ibm_db.conn_errormsg()}" )
    sys.exit(1)

#################################executions
df = pd.read_csv("transformed_weather_data.csv")
load_data_to_db2(df, table_name="weather_data")


Connected to database:  bludb as user:  MGV47022 on host:  54a2f15b-5c0f-46df-8954-7e38e612c2bd.c1ogj3sd0tgtu0lqde00.databases.appdomain.cloud
DBMS_NAME:  DB2/LINUXX8664
DBMS_VER:   11.05.0900
DB_NAME:    BLUDB


No primary key found -> table=weather_data
INFO:__main__:Data loaded into the table weather_data successfully!


Data loaded into the table weather_data successfully!


In [11]:
def db2_connection():
  # connect to IBM DB2 cloud db
  #IBM db2 config


  class PatchedConnection(ibm_db_dbi.Connection):
    @property
    def current_schema(self):
        return dsn_uid
  def get_conn():
      conn = ibm_db.connect(dsn, "", "")
      return PatchedConnection(conn)
  try:
    conn = ibm_db.connect(dsn, "", "")
    engine = create_engine("db2+ibm_db://", creator=get_conn)
    return conn, engine
  except SQLAlchemyError as e:
    print(f"SQLAlchemyError : {str(e)} ", )
  except Exception as e:
    print(f"Error loading data to IBMDB2 : {e}" )
    print(f"Error loading data to ibm_db.conn_errormsg(): {ibm_db.conn_errormsg()}" )
    sys.exit(1)

In [9]:
!pip install gspread gspread-dataframe
!pip install gspread gspread-dataframe gspread-auth


ERROR: Could not find a version that satisfies the requirement gspread-auth (from versions: none)
ERROR: No matching distribution found for gspread-auth


In [21]:
import pandas as pd
import ibm_db
import ibm_db_dbi
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

import gspread
from gspread_dataframe import set_with_dataframe

# Read from IBM DB2 cloud to export to google sheets to be used by Google Looker
def read_data_db2_gs(table_name="WEATHER_DATA"):
  try:
    conn, engine = db2_connection()
    print(conn,engine)
    print ("Connected to database: ", dsn_database, "as user: ", dsn_uid, "on host: ", dsn_hostname)
    server = ibm_db.server_info(conn)

    print ("DBMS_NAME: ", server.DBMS_NAME)
    print ("DBMS_VER:  ", server.DBMS_VER)
    print ("DB_NAME:   ", server.DB_NAME)
    # create table in remote IBM cloud DB2 database
    # createQuery = "create table weather_data(DATE DATE, temperature NUMERIC(10, 2), feels_like NUMERIC(10, 2), humidity NUMERIC(10, 2), windspeed NUMERIC(10, 2), resolvedaddress VARCHAR(50))"
    #Now fill in the name of the method and execute the statement
    # createStmt = ibm_db.exec_immediate(conn,createQuery)
    df = pd.read_sql(f"SELECT * from {table_name}", engine)
    # df = pd.read_csv("transformed_weather_data.csv")
    print(f"Data read from into the table {table_name} successfully!")
    logger.info(f"Data loaded into the table {table_name} successfully!")
    print(df)
    # df.to_csv("/content/weather_data.csv", index=False)
    # print("saved to /content/weather_data.csv ")


    # push to google sheets opens browser to login with google account no crednetials.json needed
    gc = gspread.service_account(filename='/content/sapient-cycle-300703-e8b417737ae1.json') # if gcp cloud is used use service credentials
    # gc = gspread.oauth(credentials_filename='/content/client_secret_899747170575-6vu3pmn521nsona5tih7utdpikil87sf.apps.googleusercontent.com.json')
    # gc = gspread.oauth()
    print("Connected ✅")
    sh = gc.open_by_url("https://docs.google.com/spreadsheets/d/1xf-vxmY9MJ7843oPXeUT0afhYCveMKCX2Kq5Ycw_T5w/edit?usp=sharing")

    # Creates the sheet if it doesn't exist
    # try:
    #     sh = gc.open("weather_data")
    #     print("Sheet found ✅")
    # except gspread.SpreadsheetNotFound:
    #     sh = gc.create("weather_data")
    #     print("Sheet created ✅")

    set_with_dataframe(sh.sheet1, df)

    ibm_db.close(conn)
    return f"Loaded {len(df)} rows to google sheets"
  except SQLAlchemyError as e:
    print(f"SQLAlchemyError : {str(e)} ", )
  except Exception as e:
    print(f"Error loading data to IBMDB2 : {e}" )
    print(f"Error loading data to ibm_db.conn_errormsg(): {ibm_db.conn_errormsg()}" )
    sys.exit(1)

#################################executions

read_data_db2_gs("weather_data")


<ibm_db.IBM_DBConnection object at 0x781303e032d0> Engine(db2+ibm_db://)
Connected to database:  bludb as user:  MGV47022 on host:  54a2f15b-5c0f-46df-8954-7e38e612c2bd.c1ogj3sd0tgtu0lqde00.databases.appdomain.cloud
DBMS_NAME:  DB2/LINUXX8664
DBMS_VER:   11.05.0900
DB_NAME:    BLUDB


INFO:__main__:Data loaded into the table weather_data successfully!


Data read from into the table weather_data successfully!
         date  temperature  feels_like  humidity  precipitation  windspeed  \
0  2026-03-17         48.5        47.0      33.7            0.0       19.2   

                 resolvedaddress  
0  Fort Worth, TX, United States  
Connected ✅


'Loaded 1 rows to google sheets'